In [2]:
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold
from scipy.optimize import linear_sum_assignment
import faiss

def compute_rel_l2(X, X_rec):
    return np.linalg.norm(X - X_rec, ord='fro') / (np.linalg.norm(X, ord='fro') + 1e-8)

def compute_sparsity(U):
    return 1 - (np.count_nonzero(U) / U.size)

def compute_fid(X, Y):
    mu1, mu2 = X.mean(0), Y.mean(0)
    var1, var2 = np.var(X, axis=0), np.var(Y, axis=0)
    fid = np.sum((mu1 - mu2)**2) + np.sum(var1 + var2 - 2 * np.sqrt(var1 * var2 + 1e-6))
    return float(np.maximum(fid, 0))  # ensure FID is non-negative

def deepknn_score(A, recon, k=5, use_gpu=False):
    A = A.astype(np.float32)
    recon = recon.astype(np.float32)

    if use_gpu:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatL2(A.shape[1]))
    else:
        index = faiss.IndexFlatL2(A.shape[1])

    index.add(A)
    D, _ = index.search(recon, k)
    return float(np.mean(D[:, 0]))

def compute_stability_from_projection(A, extract_fn, k_folds=5):
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    V_list = []

    for _, idx in kf.split(A):
        A_fold = A[idx]
        _, V = extract_fn(A_fold)
        V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-8)
        V_list.append(V)

    similarities = []
    for i in range(len(V_list)):
        for j in range(i + 1, len(V_list)):
            sim_matrix = cosine_similarity(V_list[i], V_list[j])
            row_ind, col_ind = linear_sum_assignment(-sim_matrix)
            matched_sim = sim_matrix[row_ind, col_ind]
            similarities.append(np.mean(matched_sim))

    return 1 - np.mean(similarities)

def evaluate_decomposition_from_pth(pth_path, extract_fn=None, method_name="Decomposition", k_folds=5, use_gpu=False):
    # Load data
    data = torch.load(pth_path)
    U = data["activations"]
    V = data["concepts"]

    if isinstance(U, torch.Tensor): U = U.cpu().numpy()
    if isinstance(V, torch.Tensor): V = V.cpu().numpy()

    # Reconstruct input A ≈ UV
    A_rec = np.dot(U, V)
    A = np.dot(U, V)  # Approximating original features as UV

    # Normalize for kNN
    A_knn = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
    A_rec_knn = A_rec / (np.linalg.norm(A_rec, axis=1, keepdims=True) + 1e-8)

    # Stability (optional)
    stability = compute_stability_from_projection(A, extract_fn, k_folds) if extract_fn else None

    # Compute all metrics
    metrics = {
        "Relative L2 ↓": compute_rel_l2(A, A_rec),
        "Sparsity ↑": compute_sparsity(U),
        "FID ↓": compute_fid(A, A_rec),
        "OOD Score ↓": deepknn_score(A_knn, A_rec_knn, k=5, use_gpu=use_gpu),
        "Stability ↓": stability,
    }

    # Print nicely
    print(f"\n=== Evaluation Results: {method_name} ===")
    for k, v in metrics.items():
        print(f"{k:<15}: {v:.4f}" if v is not None else f"{k:<15}: N/A")

    return metrics


In [16]:
import numpy as np
from sklearn.utils.extmath import randomized_svd

def extract_semi_nmf(A, n_components=3584, n_iter=100, random_state=0):
    np.random.seed(random_state)
    A = A - A.mean(axis=0)  # Optional: mean center

    # Step 1: Initialize V using truncated SVD
    U_init, S, Vt = randomized_svd(A, n_components=n_components, random_state=random_state)
    V = np.abs(Vt)  # Ensure V is non-negative

    for _ in range(n_iter):
        # Solve U = A V^T (V V^T)^-1
        VVt = V @ V.T + 1e-6 * np.eye(V.shape[0])
        U = A @ V.T @ np.linalg.inv(VVt)

        # Update V using multiplicative rule
        UUt = U.T @ U + 1e-6
        AU = A.T @ U
        V *= AU.T / (V @ UUt + 1e-6)
        V = np.clip(V, 1e-6, None)

    return U, V


In [19]:
pth_path = "/mnt/abka03/xl-vlms/results/decompose_activations_text_grounding_image_grounding_phrase_embeddings_concepts_combined.pth"

# Optional: define extractor_fn if you want to compute stability
metrics = evaluate_decomposition_from_pth(
    pth_path,
    extract_fn=lambda A: extract_semi_nmf(A, n_components=3584, n_iter=100),
    method_name="Semi-NMF"
)


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 197 is different from 3584)

In [14]:
score = "/mnt/abka03/Projects/xl-vlms/results/concept_dictionary_evaluation_overlap_clipscore_bertscore_coco15object.pth"
data = torch.load(score)
print(data.keys())
print(data['clip_score']['top_1_mean'])

dict_keys(['clip_score', 'bert_score', 'overlap_scores'])
0.6029954
